# Демо ноутбук

## Использование в колабе:

### Установка

Клонируем репозиторий из гитхаба:

In [1]:
!git clone -b asr_hw --single-branch https://github.com/ShabDarya/dla-asr

Cloning into 'dla-asr'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 226 (delta 101), reused 178 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (226/226), 58.29 KiB | 3.24 MiB/s, done.
Resolving deltas: 100% (101/101), done.


### Установка пакетов
Перейдем в папку проекта:

In [2]:
cd dla-asr

/content/dla-asr


In [3]:
!python3 -m pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.6/811.6 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 106.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Использование модели:
Конфиг лучшей модели находится в репозитории в src/configs/best_model. Веса модели хранятся на hugging face, скачаем их и сохраним к конфигу в папку.

In [4]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="Nop659/dla_hw",
    filename="model_best.pth",
    local_dir="src/configs/best_model",
)

print("Saved to:", path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_best.pth:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

Saved to: src/configs/best_model/model_best.pth


Для тестового запуска будем использовать часть датасета dev_clean. Скачаем его.

In [5]:
!wget https://www.openslr.org/resources/12/dev-clean.tar.gz

--2025-12-30 13:10:20--  https://www.openslr.org/resources/12/dev-clean.tar.gz
Resolving www.openslr.org (www.openslr.org)... 136.243.171.4
Connecting to www.openslr.org (www.openslr.org)|136.243.171.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 337926286 (322M) [application/x-gzip]
Saving to: ‘dev-clean.tar.gz’

dev-clean.tar.gz    100%[===================>] 322.27M  28.1MB/s    in 12s     

2025-12-30 13:10:32 (26.4 MB/s) - ‘dev-clean.tar.gz’ saved [337926286/337926286]



In [6]:
!mkdir -p /content/dla-asr/data/datasets
!tar -xzf dev-clean.tar.gz -C /content/dla-asr/data/datasets

### Использование модели

Для использования модели передаем аргументы from_pretrained - путь весов модели, model - конфиг модели.

Для inference.py автоматически используется конфиг в папке src/configs/inference.yaml. Также мы можем самостоятельно задавать необходимые параметры, если передадим их через аргументы (пример 2), либо если пропишем их в файле конфига (пример 1)

##### Пример 1:

In [7]:
%%writefile /content/dla-asr/src/configs/datasets/example_eval.yaml
val:
  _target_: src.datasets.LibrispeechDataset
  part: "dev-clean"
  data_dir: /content/dla-asr/data/datasets/LibriSpeech
  instance_transforms: ${transforms.instance_transforms.inference}


Overwriting /content/dla-asr/src/configs/datasets/example_eval.yaml


In [8]:
!python3 inference.py

/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = Gain(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
Preparing librispeech folders: dev-clean: 100% 97/97 [00:11<00:00,  8.54it/s]
ConformerModel(
  (input_linear): Sequential(
    (0): Linear(in_features=

##### Пример 2:

In [ ]:
!python3 inference.py datasets.val.part="test-clean"

### Расчет метрик:

Для calc_metrics.py автоматически используется конфиг в папке src/configs/metrics/calc_metrics.yaml. Аналогично можем передавать пути через файл конфига или аргументы.

In [10]:
import gdown
!gdown --folder https://drive.google.com/drive/folders/1NlD_M_myyb52WL2Ei4wz0x-O5S_RTzzT?usp=drive_link -O /content/dla-asr/example

Retrieving folder contents
Retrieving folder 1afC8nfjg1ppxkosWFuTpErQC0zEzSsWg audio
Processing file 1izlutXpwb0ry26Sgac45CcTwKwJqiZJs 84-121123-0000.flac
Processing file 1vqeO2OTxVvgF_3ISUF78Qha25102_HwT 84-121123-0001.flac
Processing file 1AulE38GNnxzdb5NI69CtYzZ_YfnhM9Ib 84-121123-0002.flac
Processing file 1VMLLKUrkLvZ5XTSfZW0ljNCrlCQQ1LWQ 84-121123-0003.flac
Processing file 1OT106fk0YBPefvviCtOoBXbx9ZvvPN83 84-121123-0004.flac
Processing file 1ADlUgTEhks_PaAOSyRt3n8y0KfWutm7A 84-121123-0005.flac
Processing file 1c2QNEskLYMUK14lbG7lirij1pElnjkto 84-121123-0006.flac
Processing file 1qBcr5yFLKHlLubU5YbIV6dOvp2Sbn10G 84-121123-0007.flac
Processing file 1BCfitLbG9CjozPjjLbCjwBn0gvnztMTd 84-121123-0008.flac
Processing file 1X8pGBSj4DhwW4KieKn137WsSrSzOUxp0 84-121123-0009.flac
Processing file 12RuamVm-fepbNqwDxkU3SPtE2VVIL-cB 84-121123-0010.flac
Processing file 1qc7NsENOGC4b1DXFRUCOldLGEG0BDJFc 84-121123-0011.flac
Processing file 1APFera24kvlWBbpP3MQrSEyUnMyQO6GX 84-121123-0012.flac
Proce

##### Пример 1:

In [11]:
%%writefile src/configs/metrics/calc_metrics.yaml
pred_path: /content/dla-asr/example/predict
trans_path: /content/dla-asr/example/one_file_one_text
one_file_one_text: True #Transcripts are in one file or one text in one file (name file is name of audio)


Overwriting src/configs/metrics/calc_metrics.yaml


In [12]:
!python3 calc_metrics.py

{'cer': 0.754302925989673, 'wer': 1.0}


##### Пример 2:

In [13]:
!python3 calc_metrics.py one_file_one_text=False trans_path=/content/dla-asr/example/one_file_many_text

{'cer': 0.754302925989673, 'wer': 1.0}


### Интерактивный пример использования:
inference.py

In [ ]:
!gdown --folder <ВАША ССЫЛКА> -O /content/<ВАШ ПУТЬ ДЛЯ СОХРАНЕНИЯ>

In [ ]:
%%writefile src/configs/datasets/custom.yaml

test:
  _target_: src.datasets.CustomDirAudioDataset
  audio_dir: /content/dla-asr/example/audio #<ПУТЬ К АУДИО>
  #transcription_dir: <ПУТЬ К ТРАНСКРИПЦИЯМ - НЕ ОБЯЗАТЕЛЕН>
  instance_transforms: ${transforms.instance_transforms.inference}

Overwriting src/configs/datasets/custom.yaml


In [ ]:
!python3 inference.py datasets=custom

/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = PitchShift(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:76: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'dict' in v0.12, and will be removed in v0.13. Make sure to update your code to something like:
  >>> augment = Gain(..., output_type='dict')
  >>> augmented_samples = augment(samples).samples
  warnings.warn(
ConformerModel(
  (input_linear): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): Dropout(p=0.1, inplace=False)
  )
 

calc_metrics.py:

audio_dir - путь к аудио, transcription_dir - путь к транскрипциям, где 1 файл = 1 транскрипция, название файла - название аудио.

In [ ]:
!python3 calc_metrics.py pred_path=<ПУТЬ К ПРЕДСКАЗАНИЯМ> trans_path=<ПУТЬ К ТРАНСКРИПЦИЯМ>

## Локальный запуск:

Все команды ниже вводятся в терминал. Код был проверен в терминале vs code.



Для локального запуска мы должны создать свою среду. Для этого проекта была использована версия питона 3.12.12.

In [ ]:
conda create -n project_env_t1 python=3.12.12

In [ ]:
conda activate project_env_t1

### Установка

Клонируем репозиторий из гитхаба:

In [ ]:
git clone -b asr_hw --single-branch https://github.com/ShabDarya/dla-asr

Устанавливаем необходимые пакеты для работы.

### Установка пакетов
Перейдем в папку проекта:

In [ ]:
cd dla-asr

c:\Users\DARYA\Desktop\Stud\DLA\test\dla-asr


In [ ]:
pip install -r requirements.txt

### Использование модели:
Конфиг лучшей модели находится в репозитории в src/configs/best_model. Веса модели хранятся на hugging face, скачаем их и сохраним к конфигу в папку. Код для скачивания находится в download_model.py. Запустим его.

In [ ]:
python download_model.py

Saved to: src\configs\best_model\model_best.pth


Для тестового запуска будем использовать часть датасета dev_clean. Скачаем его.

In [ ]:
python -c "import os, wget; os.makedirs('data/datasets', exist_ok=True); wget.download('https://www.openslr.org/resources/12/dev-clean.tar.gz', out='data/datasets/dev-clean.tar.gz')"

In [ ]:
python -c "import tarfile; tarfile.open('data/datasets/dev-clean.tar.gz','r:gz').extractall('data/datasets')"

Для использования модели передаем аргументы from_pretrained - путь весов модели, model - конфиг модели.

Для inference.py автоматически используется конфиг в папке src/configs/inference.yaml. Также мы можем самостоятельно задавать необходимые параметры, если передадим их через аргументы (пример 2), либо если пропишем их в файле конфига (пример 1)

##### Пример 1:

In [ ]:
python inference.py

^C


##### Пример 2:

In [ ]:
python inference.py inferencer.device=cuda inferencer.save_path=/content/saving

Could not override 'device'.
To append to your config use +device=cuda

Key 'device' is not in struct
    full_key: device
    object_type=dict


Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


### Расчет метрик:

Для calc_metrics.py автоматически используется конфиг в папке src/configs/metrics/calc_metrics.yaml. Аналогично можем передавать пути через файл конфига или аргументы.

In [ ]:
mkdir "example"

In [ ]:
gdown --folder https://drive.google.com/drive/folders/1NlD_M_myyb52WL2Ei4wz0x-O5S_RTzzT?usp=drive_link -O "example"

##### Пример 1:

In [ ]:
python calc_metrics.py

{'cer': 0.754302925989673, 'wer': 1.0}


##### Пример 2:

In [ ]:
python calc_metrics.py one_file_one_text=False trans_path=example/one_file_many_text

{'cer': 0.754302925989673, 'wer': 1.0}


### Интерактивный пример использования:
inference.py

In [ ]:
gdown --folder <ВАША ССЫЛКА> -O "<ВАШ ПУТЬ ДЛЯ СОХРАНЕНИЯ>"

In [ ]:
python inference.py datasets=custom datasets.test.audio_dir=<ПУТЬ К АУДИО> #datasets.test.transcription_dir=<ПУТЬ К ТРАНСКРИПЦИИ>

calc_metrics.py:

audio_dir - путь к аудио, transcription_dir - путь к транскрипциям, где 1 файл = 1 транскрипция, название файла - название аудио.

In [ ]:
python calc_metrics.py pred_path=<ПУТЬ К ПРЕДСКАЗАНИЯМ> trans_path=<ПУТЬ К ТРАНСКРИПЦИЯМ>